> 在线讲义：[第 49 章 · 请求负载下的性能与配置选择：数正确且按时的答案](https://wikieden.github.io/ai-infra-from-constraints/lessons/49-serving-choice.html) · 本书《从约束推导 AI Infra》

# 第 49 章 · 请求负载下的性能与配置选择：数正确且按时的答案

配套讲义：仓库里 `ai-infra/notes/lessons/49-serving-choice.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



Little 定律、有效吞吐、实验 8-1 的 MBU / MFU、16 条长请求四种方案、Amdahl 取自 AII 第 8 章 §8.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MiB = 2**20
READS = {1: 15.44, 4: 16.34, 16: 19.97, 64: 34.46}
STEPS = {1: 26.26, 4: 26.62, 16: 27.44, 64: 29.63}

**函数 `mbu`**：小工具。

In [ ]:
def mbu(b: int, bw: float = 1792) -> float:
    return READS[b] / bw * 1e3 / STEPS[b]

**函数 `gpu_seconds`**：小工具。

In [ ]:
def gpu_seconds(L: int, G: int, rP: float, rD: float) -> float:
    return L / rP + (G - 1) / rD

**函数 `interp_step`**：小工具。

In [ ]:
def interp_step(read_gb: float) -> float:
    return STEPS[16] + (read_gb - READS[16]) / (READS[64] - READS[16]) * (STEPS[64] - STEPS[16])

In [ ]:
PLANS = {
    "A": dict(mem=16 * 1188, prefill=7.34, rounds=255, step=29.63),
    "B": dict(mem=864 + 16 * 324, prefill=2.04, rounds=255, step=27.35),
    "C": dict(mem=16 * 1188 * 34 / 64, prefill=7.34, rounds=255, step=None),
    "D": dict(mem=864 + 16 * 324 + 3188, prefill=2.23, rounds=85, step=27.35),
}

**函数 `plan_time`**：整批耗时（s）= prefill + 轮数 × 每轮；C 每轮读 25.40 GB，按表插值。

In [ ]:
def plan_time(name: str) -> float:
    """整批耗时（s）= prefill + 轮数 × 每轮；C 每轮读 25.40 GB，按表插值。"""
    p = PLANS[name]
    step = p["step"] if p["step"] else interp_step(25.40)
    return p["prefill"] + p["rounds"] * step / 1e3

**函数 `amdahl`**：小工具。

In [ ]:
def amdahl(f: float, s: float) -> float:
    return 1 / ((1 - f) + f / s)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Little 定律")
print(f"  4 req/s × 0.5 s = {4 * 0.5:.0f}；× 2 s = {4 * 2:.0f} 条在系统内")
print(f"  336 次回答：294 正确，155 正确且按时")

In [ ]:
print("\n第二步：实验 8-1 的效率")
for b in READS:
    print(f"  batch {b:2d}：下界 {READS[b] / 1.792:.2f} ms，实测 {STEPS[b]} ms，MBU {mbu(b):.0%}")

In [ ]:
print("\n第三步：16 条 8K 长请求（前 6K 共享），12 GiB、7 s")
rP, rD = 32768 / 2.04, 16 / 0.02735
print(f"  B：r_P = {rP:,.0f} token/s，r_D = {rD:.0f} 调用/s，每条 {gpu_seconds(2048, 256, rP, rD):.3f} GPU s")
print(f"  C 的一轮（读 25.40 GB）插值 {interp_step(25.40):.2f} ms")
for k, p in PLANS.items():
    t = plan_time(k)
    ok = p["mem"] <= 12 * 1024 and t <= 7
    print(f"  {k}：内存 {p['mem'] / 1024:.2f} GiB，总耗时 {t:.2f} s {'✓' if ok else '✗'}")
print(f"  D 每个合格结果 {plan_time('D') / 16:.3f} GPU s，≤ {600 * plan_time('D') / 16:.0f} J")
print(f"  输出只要 16 个：B {2.04 + 15 * 0.02735:.2f} s，D {2.23 + 5 * 0.02735:.2f} s")

In [ ]:
print("\n第四步：完整任务")
for f in (0.5, 0.2):
    print(f"  decode 占 {f:.0%}、加速 4 倍：整体 {amdahl(f, 4):.2f} 倍")

## 3. 代码里才看得到的

1. D 代码得 4.55 s，原文写 4.56 s：2.23 + 85 × 27.35 ms = 4.5548。
2. A 的内存 16 × 1188 MiB = 18.56 GiB，按 12 GiB 直接排除，表里的时间只是"放得下时要多久"。
3. 测试里用"内存 ≤ 12 GiB 且时间 ≤ 7 s"筛选，结果只剩 D。

## 4. 自己改一改

- 加方案 E：共享前缀 + 从上下文查找草稿，每轮 2 个、128 轮、每轮多 0.1 ms，算它的位置。
- 把 `PLANS` 的步时间改成开启 CUDA Graph 后的值（假设固定开销减半），重新选。
- 用 `goodput` 写一个到达率扫描，找有效吞吐开始下降的点。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 候诊厅人数 = 来的速度 × 待的时间；
> 快递考核只数准时没送错的；送得多都迟到等于白送；
> 实测油耗才算得准，起步红灯是固定开销；
> 先看装得下，再看到得了，最后挑最省；decode 快四倍，任务只快一点几。